# TDE: Dashboard Interativo com Streamlit

**Contexto:** time de Business Intelligence de uma rede varejista. O objetivo é construir um painel completo, com filtros, métricas e gráficos, a partir de um histórico sintético de vendas, e documentar o roteiro de publicação na nuvem via Streamlit Community Cloud.

Conteúdo abordado:
- Fase 1: estrutura básica do app e cache de dados (`@st.cache_data`)
- Fase 2: layout com painel lateral e filtros interativos
- Fase 3: métricas em destaque, abas e visualização de dados
- Fase 4: roteiro de publicação (deploy) na nuvem

Este notebook gera, na mesma pasta, os três artefatos exigidos pelo enunciado: `vendas.csv`, `meu_dashboard.py` e `requirements.txt`. Como um dashboard Streamlit é uma aplicação web (não roda dentro de células de notebook), o app é executado depois via terminal: `streamlit run meu_dashboard.py`.

## Preparação: geração da base de dados (`vendas.csv`)

In [ ]:
import csv
import random
import datetime

random.seed(42)

categorias = ['Eletrônicos', 'Vestuário', 'Alimentos', 'Livros', 'Brinquedos', 'Móveis']
produtos = {
    'Eletrônicos': ['Fone de Ouvido', 'Smartphone', 'Carregador', 'Mouse', 'Teclado'],
    'Vestuário': ['Camiseta', 'Calça Jeans', 'Jaqueta', 'Tênis', 'Boné'],
    'Alimentos': ['Café', 'Chocolate', 'Biscoito', 'Suco', 'Cereal'],
    'Livros': ['Romance', 'Técnico', 'Infantil', 'Biografia', 'Quadrinho'],
    'Brinquedos': ['Boneco', 'Quebra-cabeça', 'Pelúcia', 'Carrinho', 'Jogo de Tabuleiro'],
    'Móveis': ['Cadeira', 'Mesa', 'Estante', 'Sofá', 'Luminária'],
}

start = datetime.date(2025, 1, 1)
rows = []
pid = 1000
for _ in range(600):
    dias = random.randint(0, 364)
    data = start + datetime.timedelta(days=dias)
    cat = random.choice(categorias)
    prod = random.choice(produtos[cat])
    qtd = random.randint(1, 5)
    preco_unit = round(random.uniform(15, 400), 2)
    receita = round(qtd * preco_unit, 2)
    pid += 1
    rows.append([pid, data.isoformat(), cat, prod, qtd, preco_unit, receita])

with open('vendas.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['id_pedido', 'data', 'categoria', 'produto', 'quantidade', 'preco_unitario', 'receita'])
    w.writerows(rows)

print(f'vendas.csv gerado com {len(rows)} pedidos')

## Fase 1: Estrutura Básica e Otimização de Desempenho

O Streamlit reexecuta o script inteiro a cada interação do usuário. Para evitar reler o CSV a cada clique, a função de carga é decorada com `@st.cache_data`: o resultado fica em memória e só é recalculado se os argumentos da função mudarem.

**Passo 1:** título principal com `st.title('Dashboard de Vendas')`.
**Passo 2:** função `carregar_dados()` que lê `vendas.csv` com Pandas.
**Passo 3:** decorar a função com `@st.cache_data`.

A célula abaixo cria o arquivo `meu_dashboard.py` com esse bloco inicial.

In [ ]:
with open('meu_dashboard.py', 'w', encoding='utf-8') as f:
    f.write('''import streamlit as st
import pandas as pd

st.title('Dashboard de Vendas')


@st.cache_data
def carregar_dados():
    df = pd.read_csv('vendas.csv', parse_dates=['data'])
    return df


df = carregar_dados()
''')

print(open('meu_dashboard.py', encoding='utf-8').read())

## Fase 2: Layout e Filtros Laterais (Interatividade)

Controles globais ficam no painel lateral; os resultados, no corpo principal.

**Passo 1:** painel lateral com `st.sidebar.title('Filtros')`.
**Passo 2:** `st.sidebar.multiselect('Selecione as Categorias', options=lista_de_categorias)` para escolher categorias.
**Passo 3:** regra de ouro do Streamlit: o valor retornado pelo widget filtra o DataFrame, então qualquer mudança no filtro atualiza a tela toda.

Este bloco é anexado (`'a'`) ao arquivo já criado na Fase 1.

In [ ]:
with open('meu_dashboard.py', 'a', encoding='utf-8') as f:
    f.write('''
st.sidebar.title('Filtros')

lista_de_categorias = sorted(df['categoria'].unique())
categorias_selecionadas = st.sidebar.multiselect(
    'Selecione as Categorias',
    options=lista_de_categorias,
    default=lista_de_categorias,
)

df_filtrado = df[df['categoria'].isin(categorias_selecionadas)]
''')

print(open('meu_dashboard.py', encoding='utf-8').read())

## Fase 3: Métricas em Destaque e Visualização de Dados

**Passo 1:** duas colunas proporcionais com `st.columns([1, 1])`.
**Passo 2:** `st.metric()` em cada coluna para Receita Total e Total de Pedidos.
**Passo 3:** navegação por abas com `st.tabs(['Evolução Mensal', 'Tabela de Dados'])`.
**Passo 4:** na primeira aba, receita agrupada por mês (Pandas) plotada com `st.area_chart()` — os dados precisam ser agrupados antes de plotar, senão o gráfico fica ilegível.
**Passo 5:** na segunda aba, `st.dataframe()` com o DataFrame filtrado e `st.download_button()` para exportar o recorte em CSV.

Bloco final, também anexado ao arquivo.

In [ ]:
with open('meu_dashboard.py', 'a', encoding='utf-8') as f:
    f.write('''
col1, col2 = st.columns([1, 1])

receita_calculada = df_filtrado['receita'].sum()
total_pedidos = df_filtrado['id_pedido'].nunique()

with col1:
    st.metric(label='Receita Total', value=f'R$ {receita_calculada:,.2f}')

with col2:
    st.metric(label='Total de Pedidos', value=total_pedidos)

aba1, aba2 = st.tabs(['Evolução Mensal', 'Tabela de Dados'])

with aba1:
    df_mensal = df_filtrado.copy()
    df_mensal['mes'] = df_mensal['data'].dt.to_period('M').astype(str)
    dados_agrupados = df_mensal.groupby('mes')['receita'].sum()
    st.area_chart(dados_agrupados)

with aba2:
    st.dataframe(df_filtrado, use_container_width=True)

    csv_export = df_filtrado.to_csv(index=False).encode('utf-8')
    st.download_button(
        label='Baixar CSV filtrado',
        data=csv_export,
        file_name='vendas_filtrado.csv',
        mime='text/csv',
    )
''')

print(open('meu_dashboard.py', encoding='utf-8').read())

## Fase 4: Publicação na Nuvem (Deploy)

O painel precisa ficar acessível para os tomadores de decisão, não só rodar localmente. Roteiro:

1. **Versionamento:** criar repositório no GitHub e commitar `meu_dashboard.py` e `vendas.csv`.
2. **Dependências:** incluir `requirements.txt` no mesmo repositório, listando as bibliotecas usadas (`streamlit`, `pandas`).
3. **Streamlit Community Cloud:** acessar a plataforma e conectar a conta do GitHub.
4. **Deploy:** selecionar o repositório e apontar `meu_dashboard.py` como arquivo principal. A plataforma gera um link público em poucos minutos.

A célula abaixo gera o `requirements.txt`.

In [ ]:
with open('requirements.txt', 'w', encoding='utf-8') as f:
    f.write('streamlit\npandas\n')

print(open('requirements.txt', encoding='utf-8').read())

## Como executar localmente

Com `vendas.csv`, `meu_dashboard.py` e `requirements.txt` gerados nesta pasta:

```bash
pip install -r requirements.txt
streamlit run meu_dashboard.py
```

O app abre em `http://localhost:8501`, com filtro de categorias na barra lateral, métricas de receita e pedidos, gráfico de evolução mensal e tabela exportável.